# 🧪 Optimized Bayesien Search - MAX HISTORY (1999-2026)
### Version Historique Complet (27 ans)
- **Périodes Clés** : Bulle Internet (2000), Subprimes (2008), COVID (2020), Hausse des taux (2022).
- **Réalisme** : Frais (0.1%), Marge (5%), Cash Yield (3%).
- **Filtrage** : Double SMA Actions & ETFs + ADX + ATR%.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import mlflow
mlflow.set_tracking_uri("http://localhost:5001")
print(f"🚀 Tracking synchronisé sur : {mlflow.get_tracking_uri()}")
import itertools
import random
import warnings
from datetime import datetime
from joblib import Parallel, delayed
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Setup Paths
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
sys.path.append(os.getcwd())
load_dotenv()

try:
    from src.common.setup_spark import create_spark_session
    from config.config_spark import Paths
    print("✅ Environnement initialisé.")
except Exception as e:
    print(f"❌ Erreur imports : {e}")

/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 5


✅ Environnement initialisé.


In [2]:
# 1. CHARGEMENT DES DONNÉES (MAX HISTORY)
def load_all_data():
    spark = create_spark_session("GridSearch_MaxHistory")
    try:
        # S&P 500 & ETFs (Start 1999)
        print("📡 Téléchargement S&P 500 et ETFs depuis 1999...")
        sp500_raw = yf.download('^GSPC', start="1999-01-01", progress=False)
        sp500_raw.index = pd.to_datetime(sp500_raw.index).tz_localize(None).normalize()
        
        etf_tickers = ['XLP', 'XLV', 'XLU', 'XLE', 'XLK', 'XLC', 'XLI', 'XLY', 'XLB', 'XLRE', 'TLT', 'IEF', 'HYG', 'GLD', 'VNQ']
        etfs_raw = yf.download(etf_tickers, start="1999-01-01", progress=False)
        etfs_raw.index = pd.to_datetime(etfs_raw.index).tz_localize(None).normalize()
        
        # Stocks Data Lake (Filtre 1999)
        print("📡 Lecture Delta Lake (Filtre 1999)...")
        df_bronze = spark.read.format("delta").load(Paths.SP500_STOCK_PRICES)
        stocks_raw = df_bronze.filter("date >= '1999-01-01'").select('date', 'symbol', 'adjHigh', 'adjLow', 'adjClose').toPandas()
        stocks_raw['date'] = pd.to_datetime(stocks_raw['date']).dt.tz_localize(None).dt.normalize()
        
        # Pivoting
        print("⚡ Pivotement des données...")
        s_close = stocks_raw.pivot(index='date', columns='symbol', values='adjClose').ffill()
        s_high = stocks_raw.pivot(index='date', columns='symbol', values='adjHigh').ffill()
        s_low = stocks_raw.pivot(index='date', columns='symbol', values='adjLow').ffill()
        
    finally:
        spark.stop()
        print("✅ Données chargées (Période: 1999 - Présent).")
    return sp500_raw, etfs_raw, s_close, s_high, s_low

sp500_data, etf_data, stocks_close, stocks_high, stocks_low = load_all_data()

2026-04-23 17:13:58.554 | INFO     | src.common.setup_spark:create_spark_session:22 - 🛠️ Configurant Spark avec GCS et BigQuery Jars...
26/04/23 17:13:59 WARN Utils: Your hostname, MacBook-Pro-5.local resolves to a loopback address: 127.0.0.1; using 192.168.1.1 instead (on interface en0)
26/04/23 17:13:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/forget/.ivy2/cache
The jars for the packages stored in: /Users/forget/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2c052518-68ee-4a2c-955c-292c4dd0959f;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/ml-prod-v2/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 68ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-2c052518-68ee-4a2c-955c-292c4dd0959f
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
26/04/23 17:13:59 

📡 Téléchargement S&P 500 et ETFs depuis 1999...
📡 Lecture Delta Lake (Filtre 1999)...


⚡ Pivotement des données...
✅ Données chargées (Période: 1999 - Présent).


In [3]:
etf_data

Price            Close                                                         \
Ticker             GLD        HYG        IEF        TLT        VNQ        XLB   
Date                                                                            
1999-01-04         NaN        NaN        NaN        NaN        NaN   5.945390   
1999-01-05         NaN        NaN        NaN        NaN        NaN   6.091239   
1999-01-06         NaN        NaN        NaN        NaN        NaN   6.198480   
1999-01-07         NaN        NaN        NaN        NaN        NaN   6.151291   
1999-01-08         NaN        NaN        NaN        NaN        NaN   6.361480   
...                ...        ...        ...        ...        ...        ...   
2026-04-17  445.929993  80.650002  95.930000  87.070000  96.680000  51.880001   
2026-04-20  442.089996  80.580002  95.839996  87.050003  97.029999  52.230000   
2026-04-21  429.570007  80.370003  95.419998  86.570000  95.309998  51.770000   
2026-04-22  435.260010  80.500000  95.519997  86.739998  94.519997  51.830002   
2026-04-23  434.459991  80.514999  95.605003  86.985001  95.269997  51.605000   

Price                                                                 \
Ticker             XLC        XLE         XLI         XLK        XLP   
Date                                                                   
1999-01-04         NaN   5.752037   15.013261   12.277926  14.216928   
1999-01-05         NaN   5.721218   15.323203   12.591850  14.324131   
1999-01-06         NaN   5.910000   15.652530   12.963904  14.497309   
1999-01-07         NaN   5.883030   15.516926   12.923212  14.332380   
1999-01-08         NaN   5.910000   15.633157   12.975533  14.274652   
...                ...        ...         ...         ...        ...   
2026-04-17  119.099998  55.020000  173.509995  154.350006  82.459999   
2026-04-20  118.750000  55.070000  173.899994  154.559998  82.389999   
2026-04-21  117.160004  55.869999  171.440002  154.690002  81.839996   
2026-04-22  117.879997  56.540001  171.039993  158.089996  82.110001   
2026-04-23  117.949997  56.645000  173.940002  157.610001  82.929901   

Price                                                           High  \
Ticker           XLRE        XLU         XLV         XLY         GLD   
Date                                                                   
1999-01-04        NaN   5.817026   17.397741    9.563574         NaN   
1999-01-05        NaN   5.887298   17.670237    9.724695         NaN   
1999-01-06        NaN   5.948400   17.963690   10.000897         NaN   
1999-01-07        NaN   5.927016   17.879850    9.989388         NaN   
1999-01-08        NaN   5.960621   18.099936   10.000897         NaN   
...               ...        ...         ...         ...         ...   
2026-04-17  44.480000  46.160000  148.800003  120.410004  448.700012   
2026-04-20  44.639999  45.750000  147.419998  119.870003  443.420013   
2026-04-21  43.779999  44.950001  145.919998  118.970001  440.250000   
2026-04-22  43.459999  44.869999  146.380005  118.930000  437.170013   
2026-04-23  43.825001  45.724998  145.804993  118.440002  435.291290   

Price                                                                          \
Ticker            HYG        IEF        TLT        VNQ        XLB         XLC   
Date                                                                            
1999-01-04        NaN        NaN        NaN        NaN   6.108395         NaN   
1999-01-05        NaN        NaN        NaN        NaN   6.121267         NaN   
1999-01-06        NaN        NaN        NaN        NaN   6.198480         NaN   
1999-01-07        NaN        NaN        NaN        NaN   6.421536         NaN   
1999-01-08        NaN        NaN        NaN        NaN   6.361480         NaN   
...               ...        ...        ...        ...        ...         ...   
2026-04-17  80.760002  96.050003  87.209999  96.809998  52.369999  119.480003   
2026-04-20  80.669998  95.919998  87.

In [4]:
def calculate_metrics(returns_series):
    metrics_keys = ['cagr', 'sharpe', 'sortino', 'calmar', 'max_drawdown', 'volatility']
    if returns_series.empty or returns_series.isna().all():
        return {k: 0.0 for k in metrics_keys}
    
    rets = returns_series.fillna(0.0).clip(-0.5, 0.5)
    cum_rets = (1 + rets).cumprod()
    ann = 52
    yrs = max(len(rets) / ann, 0.1)
    
    final_nav = cum_rets.iloc[-1] if not cum_rets.empty else 1.0
    cagr = (final_nav)**(1/yrs) - 1 if final_nav > 0 else -1
    
    vol = rets.std() * np.sqrt(ann)
    sharpe = (rets.mean() * ann) / vol if vol > 1e-6 else 0
    
    down_rets = rets[rets < 0]
    down_vol = (down_rets.std() * np.sqrt(ann)) if not down_rets.empty else 1e-6
    sortino = (rets.mean() * ann) / down_vol if down_vol > 1e-6 else 0
    
    m_max = cum_rets.cummax()
    dd = (cum_rets - m_max) / m_max
    m_dd = dd.min() if not dd.empty else -1
    calmar = cagr / abs(m_dd) if abs(m_dd) > 1e-6 else 0
    
    return {'cagr': float(cagr), 'sharpe': float(sharpe), 'sortino': float(sortino), 'calmar': float(calmar), 'max_drawdown': float(m_dd), 'volatility': float(vol)}

In [ ]:
class RegimeSwitchingBacktester:
    def __init__(self, config, sp500_raw, etfs_raw, s_close, s_high, s_low):
        self.config = config
        self.sp500_raw = sp500_raw
        self.etfs_raw = etfs_raw
        self.s_close = s_close
        self.s_high = s_high
        self.s_low = s_low
        
        self.fees = config.get('fees', 0.001)
        self.m_rate = config.get('margin_rate', 0.06)
        self.c_yield = config.get('cash_yield', 0.02)

    def run(self):
        # 1. REGIME SP500
        sp_p = self.sp500_raw.xs('Close', 1, 0).iloc[:,0] if isinstance(self.sp500_raw.columns, pd.MultiIndex) else self.sp500_raw.iloc[:,0]
        sp500 = pd.DataFrame(sp_p.resample('W-FRI').last().ffill())
        sp500.columns = ['Price']
        reg_f, reg_s = self.config['sp500_sma_fast'], self.config['sp500_sma_slow']
        
        sma_f = ta.trend.sma_indicator(sp500['Price'], reg_f)
        sma_s = ta.trend.sma_indicator(sp500['Price'], reg_s)
        cond_bull = ((sma_f > sma_s) & (sp500['Price'] > sma_s)) | ((sma_f < sma_s) & (sp500['Price'] > sma_f))
        sp500['Regime'] = np.where(cond_bull, 'Bull', 'Bear')
        
        # 2. ETFs
        e_close = self.etfs_raw.xs('Close', 1, 0) if isinstance(self.etfs_raw.columns, pd.MultiIndex) else self.etfs_raw
        e_close = e_close.resample('W-FRI').last().ffill()
        e_elig = pd.DataFrame(False, index=e_close.index, columns=e_close.columns)
        e_mom = e_close.pct_change(self.config['etf_mom_period'])
        
        e_ma_f = e_close.apply(lambda x: ta.trend.sma_indicator(x, self.config['etf_sma_fast']))
        e_ma_s = e_close.apply(lambda x: ta.trend.sma_indicator(x, self.config['etf_sma_slow']))
        
        for t in e_close.columns:
            e_elig[t] = (e_ma_f[t] > e_ma_s[t]) & (e_close[t] > e_ma_s[t])
            
        # 3. STOCKS
        sc = self.s_close.resample('W-FRI').last().ffill()
        sh = self.s_high.resample('W-FRI').last().ffill()
        sl = self.s_low.resample('W-FRI').last().ffill()
        
        s_elig = pd.DataFrame(False, index=sc.index, columns=sc.columns)
        s_mom = sc.pct_change(self.config['stock_mom_period'])
        
        # 👉 L'indicateur Momentum 1W pour ta condition
        s_mom_1w = sc.pct_change(1) 
        
        s_ma_f = sc.apply(lambda x: ta.trend.sma_indicator(x, self.config['stock_sma_fast']))
        s_ma_s = sc.apply(lambda x: ta.trend.sma_indicator(x, self.config['stock_sma_slow']))
        
        for t in sc.columns:
            try:
                adx = ta.trend.adx(sh[t], sl[t], sc[t], 20, fillna=True)
                atr = ta.volatility.average_true_range(sh[t], sl[t], sc[t], 14, fillna=True)
                
                cond_trend = (s_ma_f[t] > s_ma_s[t]) & (sc[t] > s_ma_s[t])
                cond_strength = adx > self.config['stock_adx_threshold']
                cond_volatility = (atr / sc[t]) < self.config['stock_atr_threshold']
                
                # La condition Pullback / 1W a été supprimée
                
                # Assemblage des filtres
                s_elig[t] = cond_trend & cond_strength & cond_volatility
                
            except: continue

        # 4. SIMULATION
        cdates = sorted(sp500.index.intersection(e_close.index).intersection(sc.index))
        rebal_freq = self.config['rebalance_freq']
        rebal_dates = set(pd.Series(cdates).groupby(pd.to_datetime(cdates).to_period(rebal_freq)).max().dt.strftime('%Y-%m-%d'))
        
        port, rets = [], []
        lev = self.config['leverage']
        prev_regime = None
        
        for i, d in enumerate(cdates):
            # --- A. Calcul des performances de la semaine ---
            if i > 0:
                w_ret = 0.0
                total_invested = 0.0
                if port:
                    pd_d = cdates[i-1]
                    for pos in port:
                        source = sc if pos['Type'] == 'Stock' else e_close
                        try:
                            r = (source.at[d, pos['Ticker']] / source.at[pd_d, pos['Ticker']]) - 1
                            if not np.isnan(r): 
                                w_ret += r * pos['Weight']
                                total_invested += pos['Weight']
                        except: pass
                
                # Coûts fixes (Cash yield si < 100%, Margin rate si > 100%)
                cash_weight = max(0, 1.0 - total_invested)
                borrowed_weight = max(0, total_invested - 1.0)
                
                w_ret += cash_weight * (self.c_yield / 52)
                w_ret -= borrowed_weight * (self.m_rate / 52)
                
                rets.append(w_ret)
            
            d_str = d.strftime('%Y-%m-%d')
            
            # --- B. STOP-LOSS HEBDOMADAIRE (Vérifié toutes les semaines) ---
            surviving_portfolio = []
            for pos in port:
                kept = True
                if pos['Type'] == 'Stock':
                    # Si le prix passe sous la SMA rapide (ex: SMA 26), on coupe
                    if sc.at[d, pos['Ticker']] < s_ma_f.at[d, pos['Ticker']]:
                        kept = False
                else:
                    # Pour l'ETF, si le prix passe sous la SMA lente (ex: SMA 50), on coupe
                    if e_close.at[d, pos['Ticker']] < e_ma_s.at[d, pos['Ticker']]:
                        kept = False
                
                if kept: surviving_portfolio.append(pos)
            
            port = surviving_portfolio

            # --- C. REBALANCEMENT (Calendrier ou Breakout) ---
            # 1. On lit le régime du jour actuel
            current_regime = sp500.at[d, 'Regime']
            
            # 2. On évalue si on vient de casser la tendance
            is_regime_breakout = (prev_regime is not None and prev_regime != current_regime)
            
            # 3. On déclenche le rebalancement si c'est la fin du mois OU si on passe en Bull
            if d_str in rebal_dates or is_regime_breakout:
                old_p = {p['Ticker']: p['Weight'] for p in port}

                # 3. On met à jour la variable 'regime' pour que la suite du code fonctionne
                regime = current_regime
                
                if current_regime == 'Bull':
                    # On garde uniquement les actions
                    port = [p for p in port if p['Type'] == 'Stock']
                    cur_t = [p['Ticker'] for p in port]
                    
                    el = s_elig.loc[d]
                    eligible_tickers = el[el].index
                    ranked = s_mom.loc[d, eligible_tickers].sort_values(ascending=False)
                    
                    # Rétention via Buffer (Top 15)
                    kept_tickers = [t for t in cur_t if t in ranked.index[:self.config['buffer_n']]]
                    
                    # Remplissage jusqu'au Top N
                    places_libres = self.config['top_n'] - len(kept_tickers)
                    if places_libres > 0:
                        cand = [t for t in ranked.index if t not in kept_tickers]
                        kept_tickers += cand[:places_libres]
                        
                    # Ajustement dynamique du poids
                    w = lev / len(kept_tickers) if kept_tickers else 0
                    new_p = [{'Ticker': t, 'Type': 'Stock', 'Weight': w} for t in kept_tickers]
                    
                else:
                    # Régime Bear : Top 2 ETFs
                    port = [p for p in port if p['Type'] == 'ETF']
                    cur_t = [p['Ticker'] for p in port]
                    
                    places_libres = 2 - len(port)
                    if places_libres > 0:
                        el = e_elig.loc[d]
                        cand = [t for t in el[el].index if t not in cur_t]
                        if cand:
                            top_e = e_mom.loc[d, cand].nlargest(places_libres).index.tolist()
                            cur_t += top_e
                            
                    # Pas de levier en Bear (Poids max total = 1.0)
                    w = 1.0 / len(cur_t) if cur_t else 0
                    new_p = [{'Ticker': t, 'Type': 'ETF', 'Weight': w} for t in cur_t]
                
                # Frais de turnover
                new_p_dict = {p['Ticker']: p['Weight'] for p in new_p}
                turnover = sum(abs(new_p_dict.get(t,0) - old_p.get(t,0)) for t in (set(old_p) | set(new_p_dict)))
                if rets: rets[-1] -= turnover * self.fees
                
                port = new_p
            
            prev_regime = regime

        return pd.Series(rets, index=cdates[1:])

In [6]:
import optuna
import mlflow

def objective(trial):
    """
    Fonction objectif que Optuna va chercher à maximiser.
    Chaque 'trial' est une suggestion intelligente de paramètres.
    """
    # 1. Définition de l'espace de recherche (Hyperparamètres)
    cfg = {
        # Paramètres SP500
        'sp500_sma_fast': trial.suggest_int('sp500_sma_fast', 10, 30, 5),
        'sp500_sma_slow': trial.suggest_int('sp500_sma_slow', 40, 75, 5),
        
        # Paramètres ETFs
        'etf_sma_fast': trial.suggest_int('etf_sma_fast', 10, 30),
        'etf_sma_slow': trial.suggest_int('etf_sma_slow', 40, 75, 5),
        'etf_mom_period': trial.suggest_categorical('etf_mom_period', [5, 13, 26, 52]),
        
        # Paramètres Actions
        'stock_mom_period': trial.suggest_categorical('stock_mom_period', [5, 13, 26, 52]),
        'stock_sma_fast': trial.suggest_int('stock_sma_fast', 10, 30, 5),
        'stock_sma_slow': trial.suggest_int('stock_sma_slow', 40, 75, 5),
        'stock_adx_threshold': trial.suggest_float('stock_adx_threshold', 0.00, 30.0, step=0.5),
        'stock_atr_threshold': trial.suggest_float('stock_atr_threshold', 0.05, 0.5, step=0.05),
        
        # Le test crucial que tu voulais faire
                
        # Gestion du risque et levier
        'leverage': trial.suggest_float('leverage', 1.0, 3.0, step=.5),
        'rebalance_freq': trial.suggest_categorical('rebalance_freq',['W', 'M', 'Q', '6M']),
        'top_n': trial.suggest_int('top_n', 5, 20, 5),
        'buffer_n': trial.suggest_int('buffer_n', 10, 50, 5),
                
        # Frais fixes
        'fees': 0.001,
        'margin_rate': 0.06,
        'cash_yield': 0.02
    }

    # 2. Exécution du Backtest avec MLflow
    # On utilise l'ID du trial pour le nom du run MLflow
    with mlflow.start_run(run_name=f"Optuna_Trial_{trial.number}", nested=True):
        mlflow.log_params(cfg)
        
        try:
            tester = RegimeSwitchingBacktester(cfg, sp500_data, etf_data, stocks_close, stocks_high, stocks_low)
            rets = tester.run()
            metrics = calculate_metrics(rets)
            
            # Enregistrement des métriques dans MLflow
            mlflow.log_metrics(metrics)
            
            # 3. Retour de la valeur à maximiser (ici le Ratio de Sharpe pour la robustesse)
            # Tu peux aussi mettre metrics['cagr'] si tu veux maximiser la performance pure
            return metrics['calmar']
            
        except Exception as e:
            print(f"❌ Trial {trial.number} échoué : {e}")
            return -1.0 # Score pénalisant en cas d'erreur

In [7]:
# ===============================================
# LANCEMENT DE L'OPTIMISATION
# ===============================================
N_TRIALS = 20 # 150 essais suffisent souvent pour explorer 9M de combinaisons
EXPERIMENT_NAME = "Momentum_Bayesian_Optimization_v2_Calmar"

mlflow.set_experiment(EXPERIMENT_NAME)

# 'study' est l'objet qui gère la mémoire de l'optimisation bayésienne
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("\n" + "="*30)
print("🏆 OPTIMISATION TERMINÉE")
print(f"Meilleur Calmar : {study.best_value:.2f}")
print("Meilleurs Paramètres :")
for key, value in study.best_params.items():
    print(f"  -> {key}: {value}")
print("="*30)

2026/04/23 17:14:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/23 17:14:35 INFO mlflow.store.db.utils: Updating database tables
2026/04/23 17:14:35 INFO mlflow.tracking.fluent: Experiment with name 'Momentum_Bayesian_Optimization_v2_Calmar' does not exist. Creating a new experiment.
[I 2026-04-23 17:14:35,351] A new study created in memory with name: no-name-0d584442-222c-4163-9b1a-6d50aa54b377
[I 2026-04-23 17:14:43,093] Trial 0 finished with value: 0.12337286725390655 and parameters: {'sp500_sma_fast': 15, 'sp500_sma_slow': 40, 'etf_sma_fast': 15, 'etf_sma_slow': 75, 'etf_mom_period': 5, 'stock_mom_period': 52, 'stock_sma_fast': 10, 'stock_sma_slow': 75, 'stock_adx_threshold': 8.5, 'stock_atr_threshold': 0.3, 'leverage': 1.0, 'rebalance_freq': 'M', 'top_n': 10, 'buffer_n': 45}. Best is trial 0 with value: 0.12337286725390655.
[I 2026-04-23 17:14:50,833] Trial 1 finished with value: 0.3490733917129903 and parameters: {'sp500_sma_fast': 15, 'sp500_sm


🏆 OPTIMISATION TERMINÉE
Meilleur Calmar : 0.35
Meilleurs Paramètres :
  -> sp500_sma_fast: 15
  -> sp500_sma_slow: 50
  -> etf_sma_fast: 14
  -> etf_sma_slow: 40
  -> etf_mom_period: 5
  -> stock_mom_period: 5
  -> stock_sma_fast: 30
  -> stock_sma_slow: 40
  -> stock_adx_threshold: 19.5
  -> stock_atr_threshold: 0.2
  -> leverage: 1.0
  -> rebalance_freq: M
  -> top_n: 20
  -> buffer_n: 20
